# Short-Term (Working) Memory — LangGraph Agent Tutorial

Short-term memory is the live message list for **one conversation thread**. In LangGraph this
is not something you hand-roll — it is what the **checkpointer** gives you for free. This
notebook builds a real LangGraph agent, compiles it with `SqliteSaver`, and shows exactly how
LangGraph scopes, persists, and (when you ask it to) trims that memory.

See `00_Memory_Layers_Guide.md` for how this layer compares to session and long-term memory.

In [1]:
# ============ IMPORTS ============
import os
import sys
import sqlite3

from langchain_core.messages import HumanMessage, trim_messages
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

sys.path.append(os.path.abspath("../../.."))
from helpers import get_llm

from dotenv import load_dotenv
load_dotenv()

print("Imports OK")

Imports OK


In [2]:
# ============ LLM INITIALIZATION ============
llm = get_llm()

LLM initialized: system.ai.gemma-3-12b (via databricks_gateway)


## 1. The Agent Graph

A single `agent` node calls the LLM with whatever is in `state["messages"]`. Nothing here
manages memory explicitly — that's the point. LangGraph's `MessagesState` + a **checkpointer**
handles persistence for us; the node only has to append the new response.

In [3]:
# ============ BUILD THE AGENT GRAPH ============
def agent_node(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

# SqliteSaver checkpoints the full message state after every node step.
DB_PATH = "short_term_memory.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent = builder.compile(checkpointer=checkpointer)
print("Short-term-memory agent compiled.")

Short-term-memory agent compiled.


## 2. Thread Scoping

Short-term memory is scoped entirely by `config["configurable"]["thread_id"]` — LangGraph never
mixes state across threads. Two invocations on the same `thread_id` share history; a new
`thread_id` starts from nothing.

In [4]:
# ============ DEMO: THREAD A REMEMBERS ITSELF ============
cfg_a = {"configurable": {"thread_id": "thread-A"}}

agent.invoke({"messages": [HumanMessage("My favorite color is teal.")]}, cfg_a)
out = agent.invoke({"messages": [HumanMessage("What color did I just say?")]}, cfg_a)
print("Thread A reply:", out["messages"][-1].content)

Thread A reply: You just said your favorite color is **teal**! 😊


In [5]:
# ============ DEMO: THREAD B IS ISOLATED ============
cfg_b = {"configurable": {"thread_id": "thread-B"}}
out = agent.invoke({"messages": [HumanMessage("What color did I say earlier?")]}, cfg_b)
print("Thread B reply:", out["messages"][-1].content)
# Thread B has never seen "teal" -- short-term memory does not leak across thread_id.

Thread B reply: You haven't mentioned a color yet in our conversation! 😊 



Perhaps you're thinking of a previous conversation we had? If so, could you remind me what we were talking about?


## 3. What the Checkpointer Actually Wrote

`SqliteSaver` is not a metaphor — it really is a SQLite table (`checkpoints`/`writes`) inside
`short_term_memory.db`. Inspecting it directly shows one row per node-step, per thread.

In [9]:
# ============ INSPECT THE CHECKPOINT TABLES ============
rows = conn.execute(
    "SELECT thread_id, COUNT(*) FROM checkpoints GROUP BY thread_id"
).fetchall()
print("checkpoint rows per thread:")
for thread_id, count in rows:
    print(f"  {thread_id}: {count} checkpoint(s)")

# get_state() reconstructs the full message list for a thread straight from the checkpointer.
state_a = agent.get_state(cfg_a)
print("\nthread-A live state has", len(state_a.values["messages"]), "messages")

checkpoint rows per thread:
  thread-A: 6 checkpoint(s)
  thread-B: 3 checkpoint(s)
  thread-long: 15 checkpoint(s)

thread-A live state has 4 messages


## 4. Trimming — Short-Term Memory Is Not Infinite

The checkpointer keeps the **full** history forever (that's what makes time-travel and resume
possible). What you *send to the model* on each turn is a separate decision — a long thread
will blow the context window and cost if you always replay everything.

The pattern below trims what's sent to the LLM inside the node, while the checkpoint itself
still records the complete, untrimmed conversation.

In [10]:
# ============ TRIMMING WHAT GOES TO THE MODEL ============
def agent_node_trimmed(state: MessagesState) -> dict:
    trimmed = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=len,       # counts messages, not tokens, for a compact demo
        max_tokens=4,            # keep only the last 4 messages sent to the model
        start_on="human",
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}

trim_builder = StateGraph(MessagesState)
trim_builder.add_node("agent", agent_node_trimmed)
trim_builder.add_edge(START, "agent")
trim_builder.add_edge("agent", END)
trimming_agent = trim_builder.compile(checkpointer=checkpointer)

cfg_long = {"configurable": {"thread_id": "thread-long"}}
for line in [
    "My name is Priya.",
    "I live in Pune.",
    "I work as a backend engineer.",
    "My favorite language is Rust.",
    "What's the first thing I told you in this chat?",
]:
    out = trimming_agent.invoke({"messages": [HumanMessage(line)]}, cfg_long)

full_state = trimming_agent.get_state(cfg_long)
print("full checkpointed history:", len(full_state.values["messages"]), "messages (nothing was dropped)")
print("model's last reply (based on only the trimmed window):")
print(" ", out["messages"][-1].content)

full checkpointed history: 20 messages (nothing was dropped)
model's last reply (based on only the trimmed window):
  The first thing you told me in this chat was: "My favorite language is Rust."


## Gotchas

- **The checkpointer is not ephemeral by default.** `SqliteSaver`/`PostgresSaver` persist to
  disk — "short-term" describes *scope* (`thread_id`), not lifetime. If you want textbook
  RAM-like behavior that dies with the process, use `langgraph.checkpoint.memory.MemorySaver`
  instead.
- **Trimming the prompt does not trim the checkpoint.** `trim_messages` (or any pre-model
  filtering inside the node) only changes what the LLM sees on this call — the full history is
  still written to `short_term_memory.db` on every step. Don't reach for it as a way to shrink
  the database; use it purely for context-window/cost control.
- **`thread_id` must come from trusted server-side config, not user text.** If a user could set
  their own `thread_id` to someone else's, thread isolation stops meaning anything.
- **SQLite concurrency.** One `sqlite3.Connection` shared with `check_same_thread=False` is fine
  for a notebook or a single-process agent; concurrent writers from multiple processes can hit
  `database is locked`. Swap `SqliteSaver` for `PostgresSaver` for real concurrent traffic.

## Key Takeaways

- Short-term memory in LangGraph = whatever the checkpointer has for a given `thread_id` —
  you don't hand-write append/replay logic, `MessagesState` + a checkpointer does it.
- `get_state(config)` lets you inspect exactly what's checkpointed for a thread at any time.
- Trimming/summarizing what reaches the model is a separate concern from what the checkpointer
  persists — keep the two straight.
